In [23]:
from groq import Groq
import os
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from dotenv import load_dotenv

from langsmith import Client

# Load environment variables from .env file
load_dotenv()
import json

In [11]:
qdrant_client = QdrantClient(url="http://localhost:6333")


In [12]:
# Check if QDRANT_URL is set
print("QDRANT_URL:", os.getenv("QDRANT_URL"))

QDRANT_URL: http://qdrant:6333


### download all data from qdrant

In [13]:
all_points=qdrant_client.scroll(collection_name="amazon-items-collection-00",
                                limit=100,
                                offset=None, with_payload=True, with_vectors=False)

In [17]:
all_points[0][0].payload

{'description': "I Am The Moon: IV. Farewell The fifth studio project from the Tedeschi Trucks band is the most ambitious and, at the same time, intimate record that America's best rock n' roll big band has ever made: a genuinely epic undertaking in four episodes and 24 songs inspired by classical literature but emotionally driven by the immediate drama, isolation and mourning of the pandemic era. CD Softpak.",
 'image': 'https://m.media-amazon.com/images/I/315JEh2Xh9L.jpg',
 'rating_number': 309,
 'price': 10.98,
 'average_rating': 4.7,
 'parent_asin': 'B09XVCWQGL'}

In [18]:
all_context=[{"id": data.payload["parent_asin"],"text":data.payload["description"]} for data in all_points[0]]

In [19]:
all_context

[{'id': 'B09XVCWQGL',
  'text': "I Am The Moon: IV. Farewell The fifth studio project from the Tedeschi Trucks band is the most ambitious and, at the same time, intimate record that America's best rock n' roll big band has ever made: a genuinely epic undertaking in four episodes and 24 songs inspired by classical literature but emotionally driven by the immediate drama, isolation and mourning of the pandemic era. CD Softpak."},
 {'id': 'B0B832LSNC',
  'text': 'When Christmas Comes Around... GRAMMY-winning global superstarKellyClarksonhas announcedWhen Christmas Comes Around…on vinyl, her ninth studio album arrived viaAtlanticRecordsin 2021. Led by the fabulously bold lead single “Christmas Isn’t Canceled (Just You),” the album explores a wide range of holiday emotions and experiences anchored by Clarkson’s incomparable vocal prowess. From the show-stopping duet “Santa, Can’t You Hear Me” featuringAriana Grandeto the hauntingly beautiful ballad “Merry Christmas (To The One I Used To Kno

### render a prompt to generate synthetic eval reference dataset

In [ ]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}


SYSTEM_PROMPT = f"""I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Also, include 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT=f"""Here is the list of chunks,each list element is a dictionary with id and text:
-DO NOT GIVE THE PREAMBLE IN THE ANSWER, JUST THE RAW JSON IN THE FORMAT SPECIFIED IN SYSTEM PROMPT
-DO NOT PUT ```json IN YOUR ANSWER, just give me the raw json that can be parsed with json.loads() in python
{all_context}
"""

In [39]:
client=Groq()

completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT
        }
        ]
    )

print(completion.choices[0].message.content)

```json
[
  {
    "question": "What is the name of the fifth studio album by Tedeschi Trucks Band?",
    "chunk_ids": ["B09XVCWQGL"],
    "answer_example": "I Am The Moon: IV. Farewell",
    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."
  },
  {
    "question": "What is the title of Kelly Clarkson's ninth studio album?",
    "chunk_ids": ["B0B832LSNC"],
    "answer_example": "When Christmas Comes Around…",
    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."
  },
  {
    "question": "What is the name of Andrew Bird's album that features the song 'Underlands'?",
    "chunk_ids": ["B0B3JV82X4"],
    "answer_example": "Inside Problems",
    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."
  },
  {
    "question": "What is the name of the album by A Day to Remember that features the song 'Right Back At it Again'?",
    "chun

In [43]:
out=completion.choices[0].message.content

In [44]:
out

'```json\n[\n  {\n    "question": "What is the name of the fifth studio album by Tedeschi Trucks Band?",\n    "chunk_ids": ["B09XVCWQGL"],\n    "answer_example": "I Am The Moon: IV. Farewell",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the title of Kelly Clarkson\'s ninth studio album?",\n    "chunk_ids": ["B0B832LSNC"],\n    "answer_example": "When Christmas Comes Around…",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of Andrew Bird\'s album that features the song \'Underlands\'?",\n    "chunk_ids": ["B0B3JV82X4"],\n    "answer_example": "Inside Problems",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of the album by A Day to Remember that features the song \'Right Bac

In [45]:
out=out[:-3]

In [46]:
out

'```json\n[\n  {\n    "question": "What is the name of the fifth studio album by Tedeschi Trucks Band?",\n    "chunk_ids": ["B09XVCWQGL"],\n    "answer_example": "I Am The Moon: IV. Farewell",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the title of Kelly Clarkson\'s ninth studio album?",\n    "chunk_ids": ["B0B832LSNC"],\n    "answer_example": "When Christmas Comes Around…",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of Andrew Bird\'s album that features the song \'Underlands\'?",\n    "chunk_ids": ["B0B3JV82X4"],\n    "answer_example": "Inside Problems",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of the album by A Day to Remember that features the song \'Right Bac

In [47]:
out=out[7:]
out

'\n[\n  {\n    "question": "What is the name of the fifth studio album by Tedeschi Trucks Band?",\n    "chunk_ids": ["B09XVCWQGL"],\n    "answer_example": "I Am The Moon: IV. Farewell",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the title of Kelly Clarkson\'s ninth studio album?",\n    "chunk_ids": ["B0B832LSNC"],\n    "answer_example": "When Christmas Comes Around…",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of Andrew Bird\'s album that features the song \'Underlands\'?",\n    "chunk_ids": ["B0B3JV82X4"],\n    "answer_example": "Inside Problems",\n    "reasoning": "The question can be answered by extracting the album title from the given chunk of text."\n  },\n  {\n    "question": "What is the name of the album by A Day to Remember that features the song \'Right Back At it

In [48]:
out=json.loads(out)

In [49]:
out

[{'question': 'What is the name of the fifth studio album by Tedeschi Trucks Band?',
  'chunk_ids': ['B09XVCWQGL'],
  'answer_example': 'I Am The Moon: IV. Farewell',
  'reasoning': 'The question can be answered by extracting the album title from the given chunk of text.'},
 {'question': "What is the title of Kelly Clarkson's ninth studio album?",
  'chunk_ids': ['B0B832LSNC'],
  'answer_example': 'When Christmas Comes Around…',
  'reasoning': 'The question can be answered by extracting the album title from the given chunk of text.'},
 {'question': "What is the name of Andrew Bird's album that features the song 'Underlands'?",
  'chunk_ids': ['B0B3JV82X4'],
  'answer_example': 'Inside Problems',
  'reasoning': 'The question can be answered by extracting the album title from the given chunk of text.'},
 {'question': "What is the name of the album by A Day to Remember that features the song 'Right Back At it Again'?",
  'chunk_ids': ['B0BG62V5J4'],
  'answer_example': 'Common Courtesy',


In [50]:
json_output=out

In [51]:
len(json_output)

28

In [59]:
def get_description(parent_asin: str) -> str:

    points = qdrant_client.scroll(
    collection_name="amazon-items-collection-00",
    scroll_filter=Filter(
    must=[
    FieldCondition(
    key="parent_asin",
    match=MatchValue(value=parent_asin)
    )
    ],
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
    )[0]

    return points[0].payload["description"]

In [60]:
get_description("B0BCBZ2WFN")

'Everyday Is Christmas Amazon Exclusive “Snowman White” Vinyl of Sia\'s first ever Christmas album. The collection of original holiday songs was written by Sia and Greg Kurstin. The album\'s lead single, "Santa\'s Coming For Us," has several counterparts on the album including other festive songs like "Candy Cane Lane," "Ho Ho Ho," and "Puppies Are Forever" which are anchored by ballads "Underneath The Christmas Lights," "Snowman," and "Snowflake."'

### create eval dataset in langsmith


In [61]:
client=Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [63]:
dataset_name="rag-evaluation-dataset"
dataset=client.create_dataset(dataset_name=dataset_name, description="Dataset for evaluation of RAG application.")

In [64]:
for item in json_output:
    client.create_example(
    dataset_id=dataset.id,
    inputs={"question": item["question"]},
    outputs={
    "ground_truth": item["answer_example"],
    "reference_context_ids": item["chunk_ids"],
    "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]}
    )